In [15]:
# Standard library
import copy
import glob
import multiprocessing
import os
import time
import zipfile

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Related third party
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [17]:
def print_model_size(mdl):
    torch.save(mdl.state_dict(), "tmp.pt")
    print("%.2f MB" %(os.path.getsize("tmp.pt")/1e6))
    os.remove('tmp.pt')

In [18]:
input_size = (224,224)
mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize(input_size),  # Resize to a fixed size
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [19]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label, folder_name in enumerate(['Dog', 'Cat']):
            folder_path = os.path.join(self.root_dir, folder_name)
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                
                try:
                    with Image.open(file_path) as img:
                        
                        if img.mode != 'RGB':
                            img = img.convert('RGB')
                        
                        if img.mode != 'RGB':
                            print(f"Skipping {file_path} because it does not have 3 channels (RGB)")
                            continue

                        self.image_paths.append(file_path)
                        self.labels.append(label)
                        
                except Exception as e:
                    print(f"Skipping {file_path} due to error: {e}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        with Image.open(image_path) as img:

            if img.mode != 'RGB':
                img = img.convert('RGB')

            if self.transform:
                img = self.transform(img)
            
        return img, label

In [20]:
dataset = CustomDataset(root_dir='../data/PetImages', transform=transform)

# Calculate split sizes
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Split dataset into train and test
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

Skipping ../data/PetImages\Dog\11702.jpg due to error: cannot identify image file '../data/PetImages\\Dog\\11702.jpg'
Skipping ../data/PetImages\Dog\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Dog\\Thumbs.db'
Skipping ../data/PetImages\Cat\666.jpg due to error: cannot identify image file '../data/PetImages\\Cat\\666.jpg'
Skipping ../data/PetImages\Cat\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Cat\\Thumbs.db'


In [21]:
# Create DataLoader for train and test sets
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([128, 3, 224, 224])
torch.Size([128])


In [22]:
def train_epoch(model, criterion, optimizer, data_loader, device,epoch):
    model.train()
    
    epoch_loss = 0.0
    num_batches = len(data_loader)
    
    for batch_idx, (image, target) in enumerate(tqdm(data_loader)):
        image, target = image.to(device), target.to(device)
        # print("before")
        output = model(image)
        # print("after")
        loss = criterion(output, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    print(f"Epoch = {epoch+1} || Training Loss: {avg_epoch_loss:.4f}")
    return avg_epoch_loss


def evaluate(model, criterion, data_loader, device,epoch):
    
    model.eval()
    
    epoch_loss = 0.0
    
    correct_predictions = 0
    total_predictions = 0
    
    num_batches = len(data_loader)
    
    with torch.no_grad():
       
        for image, target in tqdm(data_loader):
            image, target = image.to(device), target.to(device)
            output = model(image)
            loss = criterion(output, target)
            # Accumulate batch loss
            epoch_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(output, 1)  # Get the predicted class index
            correct_predictions += (predicted == target).sum().item()
            total_predictions += target.size(0)
            
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    accuracy = correct_predictions / total_predictions
    
    print(f"Epoch = {epoch+1} || Test Loss: {avg_epoch_loss:.4f} || Test Accuracy: {accuracy:.4f}")



In [23]:
class PLReLU(nn.Module):

    def __init__(self, init_alpha=0.1, init_beta=0.0):
        super().__init__()

        # learnable negative slope
        self.alpha = nn.Parameter(
            torch.tensor(init_alpha, dtype=torch.float32)
        )

        # learnable offset
        self.beta = nn.Parameter(
            torch.tensor(init_beta, dtype=torch.float32)
        )

    def forward(self, x):

        x_shifted = x - self.beta

        """
        Return the formula x_out = x - beta        if x > beta
                           x_out = alpha(x - beta) if x <= beta
        """
        return torch.where(
            x_shifted > 0,
            x_shifted,
            self.alpha * x_shifted
        )

class MobileNet(torch.nn.Module):
    def __init__(self):
        super(MobileNet, self).__init__()
        self.model = models.mobilenet_v2(weights=None)  
        
        # for param in self.model.parameters():
        #     param.requires_grad = False
            
        
        
        self.model.classifier[1] = nn.Sequential(
            nn.Linear(in_features=self.model.classifier[1].in_features,out_features=512),
            nn.LeakyReLU(negative_slope=0.02,inplace=False),
            nn.BatchNorm1d(num_features=512),
            nn.Dropout(p=0.4,inplace=False),
            nn.Linear(in_features=512,out_features=2),
            nn.Softmax(dim=1))
        
        # print(self.model)

    def forward(self, x):
        x = self.model(x)
        return x

def replace_activation(module):

    for name, child in module.named_children():

        # ---------------------------------------------
        # Replace ReLU6
        # ---------------------------------------------
        if isinstance(child, nn.ReLU6):

            setattr(
                module,
                name,
                PLReLU(
                    init_alpha=0.1,
                    init_beta=0.0
                )
            )

            # print(f"Replaced ReLU6 at: {name}")

        # ---------------------------------------------
        # Replace LeakyReLU
        # ---------------------------------------------
        elif isinstance(child, nn.LeakyReLU):

            setattr(
                module,
                name,
                PLReLU(
                    init_alpha=child.negative_slope,
                    init_beta=0.0
                )
            )

            # print(f"Replaced LeakyReLU at: {name}")

        else:
            # recursive search
            replace_activation(child)

In [10]:
model = MobileNet()
print(model)
model.load_state_dict(torch.load("../model/original_catndog_mobilenetv2.pth"))

MobileNet(
  (model): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96, kernel_size=(1, 1)

<All keys matched successfully>

In [11]:

replace_activation(model)
# model.load_state_dict(torch.load("../model/PLReLU_catndog_mobilenetv2.pth"))
print_model_size(model)
print(model)

11.78 MB
MobileNet(
  (model): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): PLReLU()
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): PLReLU()
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 

In [12]:
epoch = 10
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
for nepoch in range(epoch):
    train_epoch(model, criterion, optimizer, train_loader, device, nepoch)

# Evaluation
print("Evaluating PLReLU model...")
evaluate(model, criterion, test_loader, device, nepoch)
# Save model back to Drive
torch.save(model.state_dict(), "../model/PLReLU_catndog_mobilenetv2.pth")

100%|██████████| 157/157 [33:59<00:00, 12.99s/it]


Epoch = 1 || Training Loss: 0.3214


100%|██████████| 157/157 [33:10<00:00, 12.68s/it]


Epoch = 2 || Training Loss: 0.3203


100%|██████████| 157/157 [32:47<00:00, 12.53s/it]


...

Epoch = 9 || Training Loss: 0.3163


100%|██████████| 157/157 [40:53<00:00, 15.63s/it]


Epoch = 10 || Training Loss: 0.3159
Evaluating PLReLU model...


100%|██████████| 40/40 [03:30<00:00,  5.27s/it]

Epoch = 10 || Test Loss: 0.3229 || Test Accuracy: 0.9894


CUSTOM QUANTIZATION OF MODEL WITH PLRELU FROM FP32 TO INT8
- Insert Splitting bit method into FakeQuantize
- The flow of Quantization is FP32 -> INT8 -> INT4_SplitBit -> INT8_reconstruct -> FP32_dequantize -> forward propagation 

In [25]:
model_PLReLU = MobileNet()
replace_activation(model_PLReLU)
model_PLReLU.load_state_dict(torch.load("../model/PLReLU_catndog_mobilenetv2.pth"))
print_model_size(model_PLReLU)
print("Evaluating PLReLU model...")
evaluate(model_PLReLU, criterion, test_loader, device, nepoch)

11.78 MB
Evaluating PLReLU model...


100%|██████████| 40/40 [01:14<00:00,  1.85s/it]

Epoch = 5 || Test Loss: 0.3166 || Test Accuracy: 0.9966


In [26]:
# Custom fake quantization combined with bit splitting method
from torch.ao.quantization.fake_quantize import FusedMovingAvgObsFakeQuantize


class SplitBitFusedFakeQuantize(FusedMovingAvgObsFakeQuantize):

    def forward(self, X):

        # ---------------------------------
        # inherit from FusedMovingAvgObsFakeQuantize to compute scale and zero point using observers
        # ---------------------------------

        if self.observer_enabled[0] == 1:

            self.activation_post_process(X.detach())

            scale, zero_point = self.calculate_qparams()

            self.scale.copy_(scale)
            self.zero_point.copy_(zero_point)

        # ---------------------------------
        # fake quantization process
        # ---------------------------------

        if self.fake_quant_enabled[0] == 1:

            # FP32 -> INT8
            X_int8 = torch.round(X / self.scale + self.zero_point)

            X_int8 = torch.clamp(X_int8, self.quant_min, self.quant_max)

            # X_int8 = X_int8.to(torch.uint8)

            # ---------------------------------
            # SPLIT-BIT: Split INT8 into 2 INT4
            # //2^4 and %2^4
            # ---------------------------------
            # n, m, h, w = X_int8.shape
            # X_int8 = X_int8.view(n,m,1,h,w)
            # X_int8 = X_int8.repeat(1,1,2,1,1)
            # X_int4 = torch.zeros((n,m,2,h,w), dtype=torch.int32)

            # X_int4[:,:,0,:,:] = X_int8 // 16
            # X_int4[:,:,1,:,:] = X_int8 % 16 



            high = X_int8 // 16
            low = X_int8 % 16

            # double-channel representation
            # -------------------------------------------------
            # view(n,m,1,h,w)
            # -------------------------------------------------
            if(X_int8.dim() == 4):

                # print(X_int8.shape)
                N, C, H, W = X_int8.shape

                X_split = X_int8.view(N, C, 1, H, W)
                X_split = X_split.repeat(1, 1, 2, 1, 1)

                # =================================================
                # store nibbles
                # =================================================

                X_split[:, :, 0, :, :] = high
                X_split[:, :, 1, :, :] = low

                # ---------------------------------
                # RECONSTRUCTION
                # ---------------------------------

                high_rec = X_split[:, :, 0, :, :] * 16
                low_rec = X_split[:, :, 1, :, :]

                X_rec_int8 = high_rec + low_rec
            else:
                X_rec_int8 = X_int8

            # ---------------------------------
            # INT8 -> FP32
            # ---------------------------------

            X_dequantize = (
                X_rec_int8.float() - self.zero_point
            ) * self.scale

            # ========================================
            # STE trick
            # ========================================

            X = X + (X_dequantize - X).detach()

        return X

In [27]:
from torch.ao.quantization import QConfig
from torch.ao.quantization.qconfig_mapping import QConfigMapping
from torch.ao.quantization.fake_quantize import default_fused_per_channel_wt_fake_quant
from torch.quantization.quantize_fx import prepare_qat_fx, prepare_fx, convert_fx
from torch.ao.quantization.fx.custom_config import PrepareCustomConfig

example_inputs = (torch.randn(1, 3, 224, 224),)
custom_qconfig = QConfig(

    activation = SplitBitFusedFakeQuantize.with_args(
        observer = torch.ao.quantization.MovingAverageMinMaxObserver,
        quant_min=0,
        quant_max=255,
        dtype=torch.quint8,
        reduce_range=False
    ),
    weight = default_fused_per_channel_wt_fake_quant
)

prepare_custom_config = (
    PrepareCustomConfig()
    .set_non_traceable_module_classes(
        [PLReLU]
    )
)

qconfig_mapping = QConfigMapping()
qconfig_mapping = qconfig_mapping.set_global(custom_qconfig)
prepared_PLReLU_split_model = prepare_qat_fx(model_PLReLU, qconfig_mapping, example_inputs, prepare_custom_config=prepare_custom_config)

C:\Users\nvdao\AppData\Local\Temp\ipykernel_21092\2203969820.py:29: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_PLReLU_split_model = prepare_qat_fx(model_PLReLU, qconfig_mapping, example_inputs, prepare_custom_config=prepare_custom_config)
c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\Lib\site-packages\torch\ao\quantization\fx\utils.py:994: UserWa

In [28]:
print(prepared_PLReLU_split_model)

GraphModule(
  (activation_post_process_0): SplitBitFusedFakeQuantize(
    fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, reduce_range=False
    (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
  )
  (model): Module(
    (features): Module(
      (0): Module(
        (0): ConvBn2d(
          3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
            fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
            (activation_post_proce

In [29]:
# Train with QAT
num_epochs = 5
prepared_PLReLU_split_model = prepared_PLReLU_split_model.to(device)
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(prepared_PLReLU_split_model.parameters(), lr = 0.0001)
for nepoch in range(num_epochs):
    train_loss = train_epoch(prepared_PLReLU_split_model, criterion, optimizer, train_loader, device, nepoch)
    model_PLReLU_split_quantized = copy.deepcopy(prepared_PLReLU_split_model)
    model_PLReLU_split_quantized.to(torch.device("cpu"))
    model_PLReLU_split_quantized = convert_fx(model_PLReLU_split_quantized.eval())
    

    # Save quantized model but not scripted
    model_PLReLU_split_quantized.eval()
    torch.save(model_PLReLU_split_quantized, f"../model/int8_PLReLU_Split_catndog_mobilenetv2_epoch{nepoch}_loss{train_loss:4f}.pth")

 11%|█         | 17/157 [06:44<1:06:02, 28.30s/it]c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|██████████| 157/157 [59:28<00:00, 22.73s/it] 


Epoch = 1 || Training Loss: 0.3182


C:\Users\nvdao\AppData\Local\Temp\ipykernel_21092\3164888248.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_PLReLU_split_quantized = convert_fx(model_PLReLU_split_quantized.eval())
100%|██████████| 157/157 [53:52<00:00, 20.59s/it]


Epoch = 2 || Training Loss: 0.3194


100%|██████████| 157/157 [54:20<00:00, 20.77s/it]


Epoch = 3 || Training Loss: 0.3164


100%|██████████| 157/157 [53:33<00:00, 20.47s/it]


Epoch = 4 || Training Loss: 0.3169


100%|██████████| 157/157 [53:48<00:00, 20.56s/it]


Epoch = 5 || Training Loss: 0.3156


In [30]:
print("Evaluating quantized model...")
evaluate(model_PLReLU_split_quantized,criterion, test_loader,torch.device("cpu"),nepoch)
print_model_size(model_PLReLU_split_quantized)

Evaluating quantized model...


100%|██████████| 40/40 [04:52<00:00,  7.32s/it]

Epoch = 5 || Test Loss: 0.3184 || Test Accuracy: 0.9942
3.34 MB


In [32]:
# Save quantized model but not scripted
model_PLReLU_split_quantized.eval()
torch.save(model_PLReLU_split_quantized.state_dict(), f"../model/onlyWeight_int8_PLReLU_Split_catndog_mobilenetv2_epoch{nepoch}_loss{train_loss:4f}.pth")

8-BIT QUANTIZATION OF PLReLU MODEL USING PYTORCH QAT DEFAULT SETTING
- MODEL QUANTIZED IS PLReLU_catndog_mobilenetv2.pth

In [11]:
backend = 'onednn'
torch.backends.quantized.engine = backend
model_PLReLU = MobileNet()
replace_activation(model_PLReLU)
model_PLReLU.load_state_dict(torch.load("../model/PLReLU_catndog_mobilenetv2.pth"))
print_model_size(model_PLReLU)
print(model_PLReLU)

11.78 MB
MobileNet(
  (model): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): PLReLU()
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): PLReLU()
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96

In [12]:
# from torch.quantization.quantize_fx import prepare_fx, convert_fx,prepare_qat_fx
from torch.quantization.quantize_fx import prepare_qat_fx, prepare_fx, convert_fx
from torch.ao.quantization import QConfigMapping, get_default_qat_qconfig_mapping
from torch.ao.quantization.fx.custom_config import PrepareCustomConfig

example_inputs = (torch.randn(1, 3, 224, 224),)

qconfig_mapping = (
    get_default_qat_qconfig_mapping(backend)
    # .set_module_name("features.1.conv.1", None)
    # .set_module_name("features.2.conv.0.0", None)
    # .set_module_name("features.2.conv.2", None)
)

prepare_custom_config = (
    PrepareCustomConfig()
    .set_non_traceable_module_classes(
        [PLReLU]
    )
)

model_PLReLU.train()
prepared_model_PLReLU = prepare_qat_fx(model_PLReLU, qconfig_mapping, example_inputs, prepare_custom_config=prepare_custom_config)
print(prepared_model_PLReLU)

C:\Users\nvdao\AppData\Local\Temp\ipykernel_21092\1969876079.py:23: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_model_PLReLU = prepare_qat_fx(model_PLReLU, qconfig_mapping, example_inputs, prepare_custom_config=prepare_custom_config)


GraphModule(
  (activation_post_process_0): FusedMovingAvgObsFakeQuantize(
    fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, reduce_range=False
    (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
  )
  (model): Module(
    (features): Module(
      (0): Module(
        (0): ConvBn2d(
          3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
            fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
            (activation_post_p

In [13]:
# Train with QAT
num_epochs = 5
prepared_model_PLReLU = prepared_model_PLReLU.to(device)
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(prepared_model_PLReLU.parameters(), lr = 0.0001)
for nepoch in range(num_epochs):
    train_loss = train_epoch(prepared_model_PLReLU, criterion, optimizer, train_loader, device, nepoch)
    model_PLReLU_quantized = copy.deepcopy(prepared_model_PLReLU)
    model_PLReLU_quantized.to(torch.device("cpu"))
    model_PLReLU_quantized = convert_fx(model_PLReLU_quantized.eval())
    
    # Save quantized model but not scripted
    model_PLReLU_quantized.eval()
    torch.save(model_PLReLU_quantized, f"../model/int8_PLReLU_catndog_mobilenetv2_epoch{nepoch}_loss{train_loss:4f}.pth")

  0%|          | 0/157 [00:00<?, ?it/s]

100%|██████████| 157/157 [44:05<00:00, 16.85s/it]


Epoch = 1 || Training Loss: 0.3194


C:\Users\nvdao\AppData\Local\Temp\ipykernel_21092\2276111328.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_PLReLU_quantized = convert_fx(model_PLReLU_quantized.eval())
100%|██████████| 157/157 [41:41<00:00, 15.93s/it]


Epoch = 2 || Training Loss: 0.3194


...

Epoch = 3 || Training Loss: 0.3187


100%|██████████| 157/157 [42:33<00:00, 16.27s/it]


Epoch = 4 || Training Loss: 0.3184


100%|██████████| 157/157 [43:36<00:00, 16.67s/it]


Epoch = 5 || Training Loss: 0.3175


In [35]:
print("Evaluating quantized model...")
evaluate(model_PLReLU_quantized,criterion, test_loader,torch.device("cpu"),nepoch)
print_model_size(prepared_model_PLReLU)

Evaluating quantized model...


  0%|          | 0/40 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [36]:
# Save quantized model but not scripted
model_PLReLU_quantized.eval()
torch.save(model_PLReLU_quantized.state_dict(), f"../model/onlyWeight_int8_PLReLU_catndog_mobilenetv2_epoch{nepoch}_loss{train_loss:4f}.pth")
print(prepared_model_PLReLU)

GraphModule(
  (activation_post_process_0): FusedMovingAvgObsFakeQuantize(
    fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([0.0187]), zero_point=tensor([114], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, reduce_range=False
    (activation_post_process): MovingAverageMinMaxObserver(min_val=-2.1179039478302, max_val=2.640000104904175)
  )
  (model): Module(
    (features): Module(
      (0): Module(
        (0): ConvBn2d(
          3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
            fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([6.1000e-05, 3.3545e-03, 2.8652e-03, 3.2317e-03, 2.7754e-04, 7.5999e-04,
                    1.2384e-03, 4.9496e-04, 8.9073e-04, 6.1000e-05, 1.5687e-03, 2